# 1D-CNN IDS Model — UAV Attack Dataset
**Source:** IEEE DataPort — Live GPS Spoofing and Jamming  
**Attack Types:** Benign, GPS Jamming, GPS Spoofing  
**Features:** 80 GPS/IMU telemetry features  
**Output:** cnn_model.keras, cnn_results.json

In [1]:
# ============================================================
# cell 2 Loading data:
# WHAT: Load preprocessed dataset files for 1D-CNN model
#
# WHY:  CNN needs clean, scaled, numeric data to train.
#       Preprocessing already done — we just load results.
#       label_classes = converts numbers back to attack names
#       feature_names = needed later for SHAP and LIME
#       num_classes = tells CNN how many output neurons to use
#                     UAV_Attack=3, ISOT=10, UAVCAN=2
#
# HOW:  Step 1: np.load() → loads .npy arrays (features)
#       Step 2: pd.read_csv() → loads labels and names
#       Step 3: squeeze() → converts DataFrame to 1D array
#       Step 4: len(label_classes) → auto-detect num_classes
# ============================================================

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
import time

save_path = "/Users/miuyanhong/Desktop/replication_studies/XAI_Feature_Selection/UAV_Attack/processed/"

X_train = np.load(save_path + "X_train.npy")
X_test  = np.load(save_path + "X_test.npy")
y_train = pd.read_csv(save_path + "y_train.csv").squeeze()
y_test  = pd.read_csv(save_path + "y_test.csv").squeeze()
label_classes = pd.read_csv(save_path + "label_classes.csv").squeeze().tolist()
feature_names = pd.read_csv(save_path + "feature_names.csv").squeeze().tolist()

num_classes = len(label_classes)

print(f"X_train: {X_train.shape}")

2026-05-15 13:58:27.332120: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


X_train: (7046, 80)


In [2]:
# ============================================================
# cell 3 Reshape:
# WHAT: Reshape features from 2D to 3D for CNN input
#
# WHY:  CNN requires 3D input: (samples, timesteps, channels)
#       Our data is 2D: (samples, features)
#       We treat each feature as one timestep with 1 channel
#       Without reshape, CNN will crash with shape error
#
# HOW:  Before: (7046, 80)   ← 2D, samples × features
#       After:  (7046, 80, 1) ← 3D, samples × features × 1
#       X.shape[0] = number of samples (rows)
#       X.shape[1] = number of features (columns)
#       The final 1 = one channel (like grayscale image)
# ============================================================

X_train_cnn = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test_cnn  = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

print(f"X_train reshaped: {X_train_cnn.shape}")
print(f"X_test reshaped:  {X_test_cnn.shape}")
print("Reshape complete!")

X_train reshaped: (7046, 80, 1)
X_test reshaped:  (3021, 80, 1)
Reshape complete!


In [3]:
# ============================================================
# cell 4 Build CNN:
# WHAT: Build a 1D-CNN model architecture for IDS
#
# WHY:  1D-CNN treats features as a sequence and learns
#       local patterns between neighboring features.
#       Better than RF for detecting subtle sequential
#       patterns in network/sensor traffic data.
#       num_classes changes per dataset:
#       UAV_Attack=3, ISOT=10, UAVCAN=2
#
# HOW:  Layer 1: Conv1D(64) → learn local feature patterns
#                kernel_size=3 → looks at 3 features at once
#                relu → keeps positive values, sets negative=0
#       Layer 2: MaxPooling → reduce size by half (80→40)
#       Layer 3: Conv1D(128) → learn deeper patterns
#       Layer 4: MaxPooling → reduce again (40→20)
#       Layer 5: Flatten → convert 2D to 1D vector
#       Layer 6: Dense(128) → fully connected layer
#       Layer 7: Dropout(0.3) → randomly drop 30% neurons
#                to prevent overfitting (memorizing training data)
#       Layer 8: Dense(num_classes, softmax) → output layer
#                softmax = converts scores to probabilities
#                one probability per attack class
# ============================================================

print("Building 1D-CNN model...")

model = keras.Sequential([
    keras.layers.Input(shape=(80, 1)),
    keras.layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    keras.layers.MaxPooling1D(pool_size=2),
    keras.layers.Flatten(),
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.3),
    keras.layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer='adam',               # adaptive learning rate optimizer
    loss='sparse_categorical_crossentropy',  # for integer labels (0,1,2...)
    metrics=['accuracy']
)

model.summary()
print("CNN model built!")

Building 1D-CNN model...


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 80, 64)         │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 40, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 40, 128)        │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 20, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2560)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       327,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 353,155 (1.35 MB)

 Trainable params: 353,155 (1.35 MB)

 Non-trainable params: 0 (0.00 B)

CNN model built!


In [4]:
# ============================================================
# cell 5 Train:
# WHAT: Train the 1D-CNN on training data
#
# WHY:  fit() = model learns from X_train and y_train
#       epochs=5 = 5 full passes through training data
#       batch_size=256 = process 256 samples at a time
#       validation_split=0.1 = use 10% of train data
#       to monitor overfitting during training
#       Training time is recorded as an SE metric
#
# HOW:  Step 1: record start time
#       Step 2: model.fit() trains the CNN
#       Step 3: record end time
#       Step 4: print training summary
# ============================================================

print("Training 1D-CNN...")
start_time = time.time()

history = model.fit(
    X_train_cnn, y_train,
    epochs=5,
    batch_size=256,
    validation_split=0.1,
    verbose=1
)

cnn_train_time = round(time.time() - start_time, 2)
print(f"Training complete! Time: {cnn_train_time}s")

Training 1D-CNN...
Epoch 1/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 4s 78ms/step - accuracy: 0.8312 - loss: 0.4449 - val_accuracy: 0.9957 - val_loss: 0.0125
Epoch 2/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - accuracy: 0.9980 - loss: 0.0092 - val_accuracy: 0.9957 - val_loss: 0.0085
Epoch 3/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.9993 - loss: 0.0036 - val_accuracy: 1.0000 - val_loss: 0.0019
Epoch 4/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 71ms/step - accuracy: 0.9993 - loss: 0.0046 - val_accuracy: 1.0000 - val_loss: 0.0018
Epoch 5/5
25/25 ━━━━━━━━━━━━━━━━━━━━ 2s 70ms/step - accuracy: 0.9994 - loss: 0.0027 - val_accuracy: 1.0000 - val_loss: 0.0012
Training complete! Time: 11.91s


In [5]:
# ============================================================
# cell 6 Evaluate:
# WHAT: Evaluate CNN on unseen test data
#
# WHY:  model.predict() returns probability for each class
#       np.argmax() picks the highest probability class
#       classification_report shows F1 per attack type
#       target_names converts numbers back to attack names
#       weighted average accounts for class imbalance
#
# HOW:  Step 1: predict probabilities for all test samples
#       Step 2: argmax → pick highest probability class
#       Step 3: calculate Accuracy, Precision, Recall, F1
#       Step 4: print detailed per-class report
# ============================================================

print("Evaluating CNN model...")
start_time = time.time()

y_pred_proba = model.predict(X_test_cnn, batch_size=256, verbose=1)
y_pred = np.argmax(y_pred_proba, axis=1)

cnn_pred_time = round(time.time() - start_time, 2)

acc = accuracy_score(y_test, y_pred)
p   = precision_score(y_test, y_pred, average='weighted')
r   = recall_score(y_test, y_pred, average='weighted')
f1  = f1_score(y_test, y_pred, average='weighted')

print(f"\n=== 1D-CNN Results ===")
print(f"Accuracy:  {acc:.4f} ({acc*100:.2f}%)")
print(f"Precision: {p:.4f} ({p*100:.2f}%)")
print(f"Recall:    {r:.4f} ({r*100:.2f}%)")
print(f"F1 Score:  {f1:.4f} ({f1*100:.2f}%)")
print(f"Training time:   {cnn_train_time}s")
print(f"Prediction time: {cnn_pred_time}s")
print(f"\nDetailed Report:")
print(classification_report(y_test, y_pred, target_names=label_classes))

Evaluating CNN model...
12/12 ━━━━━━━━━━━━━━━━━━━━ 1s 32ms/step

=== 1D-CNN Results ===
Accuracy:  0.9997 (99.97%)
Precision: 0.9997 (99.97%)
Recall:    0.9997 (99.97%)
F1 Score:  0.9997 (99.97%)
Training time:   11.91s
Prediction time: 0.81s

Detailed Report:
              precision    recall  f1-score   support

 GPS_Jamming       1.00      1.00      1.00       438
GPS_Spoofing       1.00      0.99      1.00       150
      benign       1.00      1.00      1.00      2433

    accuracy                           1.00      3021
   macro avg       1.00      1.00      1.00      3021
weighted avg       1.00      1.00      1.00      3021



In [6]:
# Cell 7 Save:
import json
import os

model.save(save_path + "cnn_model.keras")

cnn_results = {
    "model": "1D-CNN",
    "dataset": "UAV_Attack",
    "accuracy": round(acc, 4),
    "precision": round(p, 4),
    "recall": round(r, 4),
    "f1_score": round(f1, 4),
    "training_time_seconds": cnn_train_time,
    "prediction_time_seconds": cnn_pred_time,
    "epochs": 5,
    "batch_size": 256
}

with open(save_path + "cnn_results.json", "w") as f:
    json.dump(cnn_results, f, indent=4)

print("Model saved: cnn_model.keras")
print("Results saved: cnn_results.json")

Model saved: cnn_model.keras
Results saved: cnn_results.json


## CNN Results Summary — UAV Attack Dataset

| Metric | Value |
|--------|-------|
| Dataset | UAV Attack (Live GPS Spoofing and Jamming) |
| Model | 1D-CNN (5 epochs, batch=256) |
| Train set | 7,046 rows |
| Test set | 3,021 rows |
| Features | 80 GPS/IMU telemetry features |
| Accuracy | 99.97% |
| Precision | 99.97% |
| Recall | 99.97% |
| F1 Score | 99.97% |
| Training time | 11.91s |
| Prediction time | 0.81s |

### Per-Class Results
| Class | F1 |
|-------|----|
| GPS_Jamming | 100% |
| GPS_Spoofing | 100% |
| benign | 100% |

### Key Finding
CNN achieved **near-perfect 99.97% F1** — consistent with RF (100%).
GPS attack signatures are highly distinctive in telemetry data.
CNN trains in only 11.91 seconds on this small dataset (7,046 rows).